## 🎯 과제 4: `/list` 명령어 추가 (실무 빈도 ★★★☆☆)

사용자가 어떤 페르소나가 있는지 매번 기억할 수 없어요. 목록 조회 명령어를 추가하세요.

**요구사항**
- `/list` 입력 시 페르소나 이름과 한 줄 소개 출력
- 예: `- 분석가: 수치와 근거 기반 답변`

In [1]:
import os

# if IS_COLAB:
#     from google.colab import userdata
#     os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# else:
from dotenv import load_dotenv
load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

✅ API 키 OK


In [2]:
# 1️⃣ OpenAI 클라이언트 생성 (한 번만 만들면 계속 재사용)
from openai import OpenAI

client = OpenAI()  # 환경변수의 OPENAI_API_KEY를 자동으로 읽어옴
print("✅ OpenAI 클라이언트 준비 완료")

✅ OpenAI 클라이언트 준비 완료


In [3]:
# 패키지 import
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

print("✅ import 완료")

✅ import 완료


In [5]:
# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "마케터": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",
    "개발자": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",
    "기획자": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",
    "디자이너": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
}

print(f"✅ {len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")

✅ 5개 페르소나 등록 완료: ['분석가', '마케터', '개발자', '기획자', '디자이너']


In [6]:
# 챗봇 핵심 함수: 페르소나 이름과 질문을 받아 답변 반환
def ask(persona_name: str, question: str) -> str:
    # 1. 페르소나 이름으로 system 메시지 조회
    system_message = PERSONAS[persona_name]

    # 2. 체인 호출 (2교시에서 배운 prompt | llm 패턴)
    result = chain.invoke({"role": system_message, "question": question})

    # 3. AIMessage 객체에서 답변 텍스트만 꺼내서 반환
    return result.content

In [ ]:
# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]
    print(cmd)
    print(parts)
    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help,/list")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")
    elif cmd == "/list":
        for name, iam in PERSONAS.items():
            print(f"- [{name}] : {iam}")
        return current_persona,False
    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

In [9]:
# 페르소나 변경 시뮬레이션
current = "분석가"
current, _ = handle_command("/persona", current)
current, _ = handle_command("/change 마케터", current)
current, _ = handle_command("/persona", current)
current, _ = handle_command("/change 외계인", current)   # 잘못된 페르소나
current, _ = handle_command("/help", current)
current, _ = handle_command("/list", current)

🎭 현재 페르소나: 분석가
✅ 페르소나가 '마케터'(으)로 변경되었습니다.
🎭 현재 페르소나: 마케터
❌ '외계인'은 등록된 페르소나가 아닙니다.
   사용 가능: 분석가, 마케터, 개발자, 기획자, 디자이너
📖 명령어: /quit, /exit, /change <이름>, /persona, /help
- [분석가] : 당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.
- [마케터] : 당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.
- [개발자] : 당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.
- [기획자] : 당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.
- [디자이너] : 당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.
✅ 5개 페르소나 등록 완료!!


In [10]:
# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")


run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

📖 명령어: /quit, /exit, /change <이름>, /persona, /help
- [분석가] : 당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.
- [마케터] : 당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.
- [개발자] : 당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.
- [기획자] : 당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.
- [디자이너] : 당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.
✅ 5개 페르소나 등록 완료!!
- [분석가] : 당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.
- [마케터] : 당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.
- [개발자] : 당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.
- [기획자] : 당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.
- [디자이너] : 당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.
✅ 5개 페르소나 등록 완료!!
👋 종료합니다.
